In [14]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ['OPENAI_API_KEY']='key'
os.environ['GROQ_API_KEY']='key'

In [16]:
model1=ChatOpenAI(
    model='gpt-5.4-mini',
    temperature=1
)

In [17]:
model2=ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=1
)

In [18]:
prompt1=PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2=PromptTemplate(
    template='Generate 5 short question ans from the following text \n {text}',
    input_variables=['text']
)

prompt3=PromptTemplate(
    template='merge the provided notes and quiz into a single document \n notes {notes} and {quiz}',
    input_variables=['notes','quiz']
)

In [19]:
parser=StrOutputParser()

In [20]:
from langchain_core.runnables  import RunnableParallel

In [21]:
parallel_chain=RunnableParallel({
    'notes':prompt1|model1|parser,
    'quiz':prompt2|model2|parser
})

merge_chain=prompt3|model1|parser

chain=parallel_chain|merge_chain

In [22]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

In [23]:
result=chain.invoke({'text':text})

In [24]:
print(result)

# SVM Notes and Quiz

## Short and simple notes on SVMs

- **SVM (Support Vector Machine)** is a supervised learning method.
- It is used for:
  - **Classification**
  - **Regression**
  - **Outlier detection**

### Advantages
- Works well in **high-dimensional spaces**.
- Good when **features are more than samples**.
- Uses only some training points called **support vectors**.
- **Memory efficient**.
- Can use different **kernel functions**.
- Custom kernels can also be used.

### Disadvantages
- Can **overfit** if features are much more than samples.
- Choosing the right **kernel** and **regularization** is important.
- Does **not directly give probability estimates**.
- Probability estimates need **expensive cross-validation**.

### Input support in scikit-learn
- Works with **dense** and **sparse** data.
- For sparse prediction, the model must be trained on sparse data.
- Best performance:
  - **Dense:** C-ordered `numpy.ndarray`
  - **Sparse:** `scipy.sparse.csr_matrix`
- Use `dty

In [25]:
chain.get_graph().print_ascii()


          +---------------------------+            
          | Parallel<notes,quiz>Input |            
          +---------------------------+            
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
  +------------+                  +----------+     
  | ChatOpenAI |                  | ChatGroq |     
  +------------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP